# FUL-105 — EDA comparativo de muestras NASA (enero de 2005)

Este notebook inspecciona una muestra auténtica de enero de 2005 por producto: MERRA-2 `T2M`, IMERG Final `Grid/precipitation` y GLDAS `SoilMoi0_10cm_inst`. Verifica tamaño/SHA contra sus manifiestos, conserva las unidades y valores crudos, calcula distribución global y reutiliza `inspect_product` y `spatial_qa` de FUL-106.

**Alcance:** un mes y un granule por producto; no es una serie temporal, climatología, análisis de tendencia ni comparación directa de magnitudes. La selección Murray–Darling cuenta centros de celda, no cobertura de superficie ni media de cuenca. Los candidatos QA de IMERG se informan, pero no se filtran. Requiere los archivos NASA y el polígono local, que no se versionan.

## 1. Rutas y verificación de procedencia

Ejecutar desde cualquier directorio dentro del checkout. Si los archivos están en otra ubicación autorizada, ajustar `INPUTS` y `BOUNDARY` aquí. La geometría se puede recuperar siguiendo la procedencia y licencia descritas en `docs/region_murray_darling.md`.

In [ ]:
import hashlib
import json
import sys
from importlib import import_module
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display


def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("No se encontró pyproject.toml desde el directorio actual")


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

backend_samples = import_module("backend.app.data_sources.earthdata_samples")
backend_spatial = import_module("backend.app.data_sources.earthdata_spatial")
inspect_product = backend_samples.inspect_product
spatial_qa = backend_spatial.spatial_qa
INPUTS = {
    "merra2": {
        "product": "merra2",
        "file": PROJECT_ROOT / "data/raw/nasa/merra2/MERRA2_300.tavgM_2d_slv_Nx.200501.nc4",
    },
    "imerg_final": {
        "product": "imerg_final",
        "file": PROJECT_ROOT / "data/raw/nasa/imerg_final/3B-MO.MS.MRG.3IMERG.20050101-S000000-E235959.01.V07B.HDF5",
    },
    "gldas_noah_monthly": {
        "product": "gldas_noah_monthly",
        "file": PROJECT_ROOT / "data/raw/nasa/gldas/GLDAS_NOAH025_M.A200501.021.nc4",
    },
}
BOUNDARY = PROJECT_ROOT / "data/cache/ful-105/mdba_boundary_4326.geojson"

# El GLDAS histórico se conservó en el checkout anterior; localízalo sin copiarlo ni modificarlo.
if not INPUTS["gldas_noah_monthly"]["file"].is_file():
    alternate = PROJECT_ROOT.parent / "hackNasa2026/data/raw/nasa/gldas/GLDAS_NOAH025_M.A200501.021.nc4"
    if alternate.is_file():
        INPUTS["gldas_noah_monthly"]["file"] = alternate

missing = [str(item["file"]) for item in INPUTS.values() if not item["file"].is_file()]
if not BOUNDARY.is_file():
    missing.append(str(BOUNDARY))
if missing:
    raise FileNotFoundError("Faltan insumos auténticos requeridos:\n" + "\n".join(missing))


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

print("Project root:", PROJECT_ROOT)
print("Boundary SHA-256:", sha256(BOUNDARY))

## 2. Lectura de valores crudos y QA estadístico

`inspect_product` aporta identidad y metadatos sin aplicar escalas ni conversiones. La lectura de arrays debajo solo extrae el campo para este EDA. Se enmascaran valores no finitos y sentinelas declarados; cero sigue siendo un valor válido. Se reportan estadísticas de la grilla global nativa, no de Murray–Darling.

In [ ]:
def read_raw_grid(key, path, report):
    if key == "imerg_final":
        with h5py.File(path, "r") as h5:
            variable = h5["Grid/precipitation"]
            labels = list(report["variable"]["dimensions"])
            data = np.asarray(variable[...])
            time_axes = [i for i, label in enumerate(labels) if label == "time"]
            for axis in reversed(time_axes):
                data = np.take(data, 0, axis=axis)
                labels.pop(axis)
            if labels == ["lon", "lat"]:
                data = data.T
            elif labels != ["lat", "lon"]:
                raise ValueError(f"Ejes IMERG no soportados: {labels}")
            lat = np.asarray(h5["Grid/lat"][...]).reshape(-1)
            lon = np.asarray(h5["Grid/lon"][...]).reshape(-1)
    else:
        with xr.open_dataset(
            path, engine="netcdf4", decode_cf=False,
            mask_and_scale=False, decode_times=False
        ) as ds:
            variable = ds[report["variable"]["name"]]
            lat_name = next(name for name in ("lat", "latitude") if name in ds.coords)
            lon_name = next(name for name in ("lon", "longitude") if name in ds.coords)
            lat_dim, lon_dim = ds[lat_name].dims[0], ds[lon_name].dims[0]
            indexers = {dim: 0 for dim in variable.dims if dim not in {lat_dim, lon_dim}}
            data = np.asarray(variable.isel(indexers).transpose(lat_dim, lon_dim).values)
            lat = np.asarray(ds[lat_name].values).reshape(-1)
            lon = np.asarray(ds[lon_name].values).reshape(-1)
    if data.shape != (lat.size, lon.size):
        raise ValueError(f"Grilla {key}: shape {data.shape} no coincide con lat/lon")
    return data, lat, lon


reports, grids, global_rows, provenance_rows = {}, {}, [], []
for key, config in INPUTS.items():
    path = config["file"]
    manifest_path = Path(str(path) + ".manifest.json")
    if not manifest_path.is_file():
        raise FileNotFoundError(f"Falta el manifiesto: {manifest_path}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    report = inspect_product(path, config["product"])
    digest = sha256(path)
    integrity_ok = manifest.get("bytes") == path.stat().st_size and manifest.get("sha256") == digest
    if not integrity_ok:
        raise ValueError(f"Integridad no coincide con el manifiesto: {path.name}")
    data, lat, lon = read_raw_grid(key, path, report)
    variable = report["variable"]
    sentinels = []
    for item in (variable.get("fill_value"), variable.get("missing_value")):
        if item is not None:
            try:
                sentinels.append(float(item))
            except (TypeError, ValueError):
                pass
    non_finite = ~np.isfinite(data)
    fill_mask = np.zeros(data.shape, dtype=bool)
    for sentinel in sentinels:
        fill_mask |= np.isclose(data, sentinel, rtol=1e-6, atol=1e-8)
    valid = np.isfinite(data) & ~fill_mask
    values = data[valid].astype(np.float64, copy=False)
    if values.size == 0:
        raise ValueError(f"No hay datos válidos para {key}")
    dx = float(np.median(np.abs(np.diff(lon)))) if lon.size > 1 else np.nan
    dy = float(np.median(np.abs(np.diff(lat)))) if lat.size > 1 else np.nan
    grids[key] = {"data": data, "valid": valid, "lat": lat, "lon": lon, "report": report, "manifest": manifest}
    reports[key] = report
    global_rows.append({
        "product": key, "variable": variable["name"], "unit": variable.get("units"),
        "shape": " × ".join(map(str, variable["shape"])),
        "native_resolution_lon_lat_deg": f"{dx:.3f} x {dy:.3f}",
        "cells_total": int(data.size), "valid": int(valid.sum()),
        "fill_or_missing": int((fill_mask & ~non_finite).sum()),
        "non_finite": int(non_finite.sum()), "zeros_valid": int(np.count_nonzero(values == 0)),
        "min": float(values.min()), "p05": float(np.percentile(values, 5)),
        "p25": float(np.percentile(values, 25)), "median": float(np.median(values)),
        "mean": float(np.mean(values)), "p75": float(np.percentile(values, 75)),
        "p95": float(np.percentile(values, 95)), "max": float(values.max()),
        "qa_candidates_reported": "; ".join(report.get("quality_variable_candidates", [])) or "none found",
        "qa_filter_applied": False,
    })
    provenance_rows.append({
        "product": key, "filename": path.name, "period": manifest.get("period", {}).get("month"),
        "bytes": path.stat().st_size, "sha256": digest,
        "manifest_matches_size_and_sha": integrity_ok,
        "collection_id": manifest.get("collection_id"), "granule_id": manifest.get("granule_id"),
        "format": report.get("format"), "variable": variable["name"], "unit": variable.get("units"),
        "dimensions": report.get("dimensions"), "shape": variable.get("shape"),
        "fill_value": variable.get("fill_value"), "missing_value": variable.get("missing_value"),
        "scale_factor": variable.get("scale_factor"), "add_offset": variable.get("add_offset"),
    })

global_summary = pd.DataFrame(global_rows).set_index("product")
provenance = pd.DataFrame(provenance_rows).set_index("product")
display(provenance)
display(global_summary)

## 3. Conversión explícita opcional de temperatura

MERRA-2 `T2M` es temperatura del aire a 2 m, originalmente en Kelvin. Esta celda calcula una vista derivada en °C para lectura humana; no altera el array original ni su unidad de procedencia.

In [ ]:
merra_valid_k = grids["merra2"]["data"][grids["merra2"]["valid"]].astype(np.float64)
merra_valid_c = merra_valid_k - 273.15
print({
    "derived_variable": "T2M_C = T2M_K - 273.15",
    "raw_unit_preserved": reports["merra2"]["variable"]["units"],
    "derived_unit": "degC",
    "n": int(merra_valid_c.size),
    "min_C": float(merra_valid_c.min()),
    "median_C": float(np.median(merra_valid_c)),
    "max_C": float(merra_valid_c.max()),
})
print("IMERG queda en mm/h; no se convierte a mm/mes. GLDAS permanece en kg m-2.")

## 4. QA espacial por centros de celda

Se reutiliza el lector FUL-106 con el polígono completo. Los conteos son de centros de celda, en grilla nativa. No se calcula intersección fraccional, ponderación por área, filtro científico de IMERG ni promedio regional.

In [ ]:
spatial_rows = []
for key, config in INPUTS.items():
    result = spatial_qa(
        config["file"], Path(str(config["file"]) + ".manifest.json"),
        BOUNDARY, config["product"], max_rows=1,
    )
    spatial_rows.append({
        "product": key,
        "polygon_sha256": result["polygon_sha256"],
        "selection": result["selection"],
        "denominator_center_cells": result["denominator_center_cells"],
        "valid_center_cells": result["valid_center_cells"],
        "missing_center_cells": result["missing_center_cells"],
        "valid_percent_of_centers": result["valid_percent"],
        "surface_area_coverage_claimed": result["surface_area_claimed"],
        "qa_filter_applied": False,
    })
spatial_summary = pd.DataFrame(spatial_rows).set_index("product")
display(spatial_summary)

## 5. Distribuciones y mapas de las grillas originales

Los histogramas y boxplots describen cada variable por separado y conservan su unidad nativa. Los boxplots usan bigotes P05–P95 y omiten puntos extremos solo en el dibujo; la tabla conserva los rangos completos. Los mapas muestran el dominio global de cada granule; los faltantes aparecen en gris. Las grillas tienen resoluciones distintas y no se remuestrean ni se combinan numéricamente.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (key, row) in zip(axes, global_summary.iterrows()):
    grid = grids[key]
    values = grid["data"][grid["valid"]]
    ax.hist(values, bins=70, color="#2878B5", edgecolor="white", linewidth=0.25)
    ax.set_title(f"{key}\n{row['variable']}")
    ax.set_xlabel(f"Valor crudo ({row['unit']})")
    ax.set_ylabel("Celdas válidas")
    ax.grid(axis="y", alpha=0.2)
fig.suptitle("Distribuciones globales — muestras NASA de enero de 2005")
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (key, row) in zip(axes, global_summary.iterrows()):
    grid = grids[key]
    values = grid["data"][grid["valid"]]
    ax.boxplot(values, vert=False, whis=(5, 95), showfliers=False, patch_artist=True)
    ax.set_title(f"{key}\n{row['variable']}")
    ax.set_xlabel(f"Valor crudo ({row['unit']})")
    ax.set_yticks([])
    ax.grid(axis="x", alpha=0.2)
fig.suptitle("Boxplots globales — enero de 2005; bigotes P05–P95, extremos en la tabla")
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (key, row) in zip(axes, global_summary.iterrows()):
    grid = grids[key]
    data = np.ma.masked_where(~grid["valid"], grid["data"])
    lat, lon = grid["lat"], grid["lon"]
    if lat[0] > lat[-1]:
        data, lat = data[::-1, :], lat[::-1]
    if lon[0] > lon[-1]:
        data, lon = data[:, ::-1], lon[::-1]
    cmap = plt.get_cmap("viridis").copy()
    cmap.set_bad("#d9d9d9")
    image = ax.imshow(
        data, origin="lower", aspect="auto", cmap=cmap,
        extent=(float(lon.min()), float(lon.max()), float(lat.min()), float(lat.max())),
    )
    ax.set_title(f"{key} — {row['variable']} ({row['unit']})")
    ax.set_xlabel("Longitud (°)")
    ax.set_ylabel("Latitud (°)")
    fig.colorbar(image, ax=ax, shrink=0.78, label=str(row["unit"]))
fig.suptitle("Grillas nativas globales — enero de 2005; no son promedio ni máscara de cuenca")
fig.tight_layout()
plt.show()

## 6. Interpretación y límites

- La muestra contiene solo enero de 2005 por producto; no permite inferencia temporal ni tendencia.
- Los conteos espaciales describen celdas cuyo centro cae dentro del polígono. No equivalen a porcentaje de superficie cubierta.
- `T2M` representa temperatura del aire a 2 m; °C es una variable derivada explícita. `precipitation` permanece como tasa mensual en mm/h. `SoilMoi0_10cm_inst` permanece en kg m⁻².
- IMERG reporta `precipitationQualityIndex` y `randomError` como candidatos. Este EDA solo los identifica; no aplica filtros porque la regla aún requiere revisión científica.
- Los histogramas y rangos globales pueden incluir valores extremos válidos; no se eliminaron observaciones.
- El plan de serie mensual e inferencia permanece en `docs/ful-105-plan-inferencia.md`. La revisión de identidad, QA y método espacial corresponde a Lucas/FUL-104.